# AIQuant — HFT Statistical Arbitrage
**AegisFintech · Apache 2.0 · [github.com/AegisFintech/AIQuant](https://github.com/AegisFintech/AIQuant)**

This notebook runs the full AIQuant pipeline in Google Colab:
- **Backtest** using historical 1m OHLCV data from [CryptoDataDownload](https://www.cryptodatadownload.com/) (free, no API key)
- **Live trading** via [Hyperliquid](https://hyperliquid.xyz) mainnet (requires private key in Step 2)

> **Note:** CryptoDataDownload files are static snapshots (BTC data ends ~July 2022). The backtest uses the last N days of available data.

| Colab Tier | RAM | Recommended Days |
|---|---|---|
| Free | 12 GB | 30 days |
| Pro | 25 GB | 90 days |
| Pro+ | 52 GB | 180 days |

## Step 1 — Clone Repository & Install Dependencies

In [ ]:
import os

# Always pull the latest version from GitHub
os.chdir('/content')
!rm -rf AIQuant
!git clone https://github.com/AegisFintech/AIQuant.git
os.chdir('/content/AIQuant')

# Show which commit we are on
!git log --oneline -3
print('\n✓ Repository cloned')

In [ ]:
# Install all dependencies
!pip install -q -r requirements.txt

import numpy as np, pandas as pd, backtrader as bt, numba
print(f'NumPy      {np.__version__}')
print(f'Pandas     {pd.__version__}')
print(f'Backtrader {bt.__version__}')
print(f'Numba      {numba.__version__}')
print('\n✓ All packages ready')

## Step 2 — Configuration

Edit the cell below. Only `HYPERLIQUID_PRIVATE_KEY` is required for live trading — leave it empty for backtesting only.

In [ ]:
import os

# ── Trading pair ─────────────────────────────────────────────────────────
# Supported: BTCUSDT, ETHUSDT, SOLUSDT, BNBUSDT, XRPUSDT
# Shorthand accepted: 'BTC', 'ETH', 'SOL' etc.
PAIR = 'BTCUSDT'

# ── Backtest settings ────────────────────────────────────────────────────
DAYS             = 30        # Free Colab: 30 | Pro: 90 | Pro+: 180
BACKTEST_CAPITAL = 100_000   # Starting capital in USD

# ── Live trading settings ────────────────────────────────────────────────
# Generate a key: python3 -c "from eth_account import Account; a=Account.create(); print(a.key.hex())"
# Fund at: https://app.hyperliquid.xyz (deposit USDC via Arbitrum)
HYPERLIQUID_PRIVATE_KEY = ''   # <-- paste your key here
LIVE_CAPITAL            = 10_000
POLL_INTERVAL_SEC       = 60

# ── Write .env file ──────────────────────────────────────────────────────
with open('/content/AIQuant/.env', 'w') as f:
    f.write(f'HYPERLIQUID_PRIVATE_KEY={HYPERLIQUID_PRIVATE_KEY}\n')
    f.write(f'DEFAULT_PAIR={PAIR}\n')
    f.write(f'DEFAULT_DAYS={DAYS}\n')
    f.write(f'BACKTEST_CAPITAL={BACKTEST_CAPITAL}\n')
    f.write(f'LIVE_CAPITAL={LIVE_CAPITAL}\n')
    f.write('KELLY_FRACTION=0.5\n')
    f.write(f'POLL_INTERVAL_SEC={POLL_INTERVAL_SEC}\n')
    f.write('MAX_POSITION_FRACTION=0.25\n')
    f.write('MAX_DAILY_LOSS=0.03\n')
    f.write('MAX_DRAWDOWN=0.15\n')
    f.write('LOG_LEVEL=INFO\n')
    f.write('LOG_DIR=logs\n')

print(f'✓ Config: {PAIR} · {DAYS} days · ${BACKTEST_CAPITAL:,} capital')
if HYPERLIQUID_PRIVATE_KEY:
    print('✓ Hyperliquid key set — live trading enabled')
else:
    print('ℹ  No Hyperliquid key — backtest mode only')

## Step 3 — Fetch Historical Data (CryptoDataDownload)

Streams the last N days of 1m OHLCV data from CryptoDataDownload. No API key required.  
Data is cached to `data/raw/` as a parquet file — subsequent runs load instantly.

> **Why not Binance?** Binance blocks Google Cloud IP ranges (used by Colab). CryptoDataDownload is a reliable free alternative.

In [ ]:
import sys, os, time
sys.path.insert(0, '/content/AIQuant')
os.chdir('/content/AIQuant')

import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)-7s | %(message)s',
                    datefmt='%H:%M:%S')

from aiquant.data.fetcher import fetch_cdd_backtest
from pathlib import Path

print(f'Fetching {PAIR} 1m data ({DAYS} days = {DAYS * 1440:,} bars)...')
print('Source: CryptoDataDownload (free, no API key)')
print('Note: CDD is a static snapshot — BTC data ends ~July 2022\n')

df_raw = fetch_cdd_backtest(
    pair=PAIR,
    days=DAYS,
    cache_dir=Path('data/raw'),
    force=False
)

print(f'\n✓ Loaded {len(df_raw):,} bars')
print(f'  Date range: {df_raw.index[0].strftime("%Y-%m-%d %H:%M")} → {df_raw.index[-1].strftime("%Y-%m-%d %H:%M")} UTC')
df_raw.tail()

## Step 4 — Feature Engineering (100+ Features)

In [ ]:
from aiquant.features.technical import generate_all_technical_features
from aiquant.features.microstructure import generate_all_microstructure_features
from aiquant.features.statarb import generate_all_statarb_features

print('Building features...')
t0 = time.time()

df_feat = generate_all_technical_features(df_raw.copy())
print(f'  Technical:      {df_feat.shape[1]} features')

df_feat = generate_all_microstructure_features(df_feat)
print(f'  Microstructure: {df_feat.shape[1]} features')

df_feat = generate_all_statarb_features(df_feat)
print(f'  StatArb:        {df_feat.shape[1]} features')

elapsed = time.time() - t0
df_clean = df_feat.dropna()
print(f'\n✓ Feature matrix: {df_clean.shape[0]:,} bars × {df_clean.shape[1]} features in {elapsed:.1f}s')
print(f'  Memory: {df_clean.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## Step 5 — Signal Generation

In [ ]:
from aiquant.strategies.ensemble import StrategyEnsemble

ensemble   = StrategyEnsemble()
signals_df = ensemble.generate_signals(df_clean)

final   = signals_df['final_signal']
n_long  = int((final == 1).sum())
n_short = int((final == -1).sum())
n_flat  = int((final == 0).sum())

print(f'✓ Signals over {len(final):,} bars')
print(f'  Long  (▲): {n_long:,}  ({n_long/len(final)*100:.1f}%)')
print(f'  Short (▼): {n_short:,}  ({n_short/len(final)*100:.1f}%)')
print(f'  Flat  (—): {n_flat:,}  ({n_flat/len(final)*100:.1f}%)')

## Step 6 — Backtest (via run.py CLI)

In [ ]:
import subprocess, sys

print(f'Running: python3 run.py backtest --pair {PAIR} --days {DAYS} --capital {BACKTEST_CAPITAL}')
print('─' * 60)

result = subprocess.run(
    [sys.executable, 'run.py', 'backtest',
     '--pair',    PAIR,
     '--days',    str(DAYS),
     '--capital', str(BACKTEST_CAPITAL)],
    capture_output=True, text=True, cwd='/content/AIQuant'
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

## Step 7 — View Backtest Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

chart_path = Path('/content/AIQuant/results/backtest_results.png')

if chart_path.exists():
    img = mpimg.imread(str(chart_path))
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Chart not found — run Step 6 first.')

## Step 8 — Live Trading on Hyperliquid

Requires `HYPERLIQUID_PRIVATE_KEY` set in Step 2 and a funded Hyperliquid account.

**Fund your account:** [app.hyperliquid.xyz](https://app.hyperliquid.xyz) (deposit USDC via Arbitrum bridge)

> ⚠️ This executes **real trades** on Hyperliquid mainnet. Only run with capital you are willing to risk.
> Press the **■ Stop** button in Colab to halt the trading loop.

In [ ]:
if not HYPERLIQUID_PRIVATE_KEY:
    print('⚠  No HYPERLIQUID_PRIVATE_KEY set in Step 2.')
    print('   Add your key, re-run Step 2, then run this cell.')
else:
    print(f'Starting live trading: {PAIR} · ${LIVE_CAPITAL:,} · poll every {POLL_INTERVAL_SEC}s')
    print('Press ■ Stop to halt.')
    print('─' * 60)
    !python3 run.py live --pair {PAIR} --capital {LIVE_CAPITAL} --poll {POLL_INTERVAL_SEC}

## Step 9 — Download Results

In [ ]:
from google.colab import files
from pathlib import Path
import glob

chart = Path('/content/AIQuant/results/backtest_results.png')
if chart.exists():
    files.download(str(chart))
    print('✓ Downloading backtest chart')
else:
    print('No chart — run Step 6 first')

for log in glob.glob('/content/AIQuant/logs/**/*.json', recursive=True):
    files.download(log)
    print(f'✓ Downloading {log}')

## Refresh — Pull Latest Code from GitHub

Run this cell any time to pull the latest version of AIQuant and re-run from Step 3.

In [ ]:
import os
os.chdir('/content')
!rm -rf AIQuant
!git clone https://github.com/AegisFintech/AIQuant.git
os.chdir('/content/AIQuant')
!git log --oneline -3
print('\n✓ Latest code pulled — re-run from Step 3')

---
**AIQuant** · Built by [AegisFintech](https://github.com/AegisFintech) · [Apache 2.0](https://github.com/AegisFintech/AIQuant/blob/main/LICENSE)

*For research and educational purposes only. Not financial advice. Trading cryptocurrencies involves significant risk of loss.*